# 03 -- Feature Selection

**Purpose:** Identify the most predictive features using LASSO (LassoCV) and RFE (GradientBoostingRegressor), then build a consensus feature set.

| Step | Description |
|---|---|
| 1 | Why feature selection matters |
| 2 | LASSO -- rank by absolute coefficient |
| 3 | RFE -- rank by recursive elimination |
| 4 | Consensus feature set |
| 5 | Model score before vs after selection |

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import RFE
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

from src.config import (PROCESSED_DATA_PATH, IMAGES_DIR, TARGET_COL,
                        TARGET_LOG_COL, RANDOM_STATE, TEST_SIZE)

pd.set_option('display.max_columns', None)

## 1. Why Feature Selection Matters

With 80+ features after one-hot encoding, we risk:
- **Multicollinearity** -- correlated features inflate variance
- **Overfitting** -- noise features hurt generalization
- **Slower training** -- unnecessary computation

We use two complementary methods and take their **consensus** for robustness.

In [2]:
df = pd.read_csv(PROCESSED_DATA_PATH)
feature_cols = [c for c in df.columns if c not in [TARGET_COL, TARGET_LOG_COL]]
X = df[feature_cols]
y = df[TARGET_LOG_COL]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

print(f"Features: {X.shape[1]}")
print(f"Train: {X_train.shape}  Test: {X_test.shape}")

Features: 55
Train: (7307, 55)  Test: (1827, 55)


## 2. LASSO Feature Ranking (LassoCV)

In [3]:
lasso = LassoCV(cv=5, max_iter=5000, random_state=RANDOM_STATE)
lasso.fit(X_train_sc, y_train)
print(f"Best alpha: {lasso.alpha_:.6f}")

lasso_coef = pd.Series(np.abs(lasso.coef_), index=feature_cols)
lasso_selected = lasso_coef[lasso_coef > 0].sort_values(ascending=False)
print(f"\nLASSO selected {len(lasso_selected)} / {len(feature_cols)} features")
print("\nTop 20 by coefficient magnitude:")
print(lasso_selected.head(20))

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
lasso_selected.head(25).sort_values().plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('LASSO -- Top Feature Importances (|coefficient|)')
ax.set_xlabel('|Coefficient|')
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'lasso_feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: lasso_feature_importance.png")

Best alpha: 0.001338

LASSO selected 44 / 55 features

Top 20 by coefficient magnitude:
Income_per_Policy                 0.432961
Income                            0.363355
Monthly Premium Auto              0.249817
Number of Policies                0.066079
Vehicle Class_SUV                 0.051621
Renew Offer Type_Offer2           0.046826
Renew Offer Type_Offer4           0.035325
Number of Open Complaints         0.031236
Coverage_Extended                 0.028500
Vehicle Class_Sports Car          0.025029
EmploymentStatus_Employed         0.022993
Vehicle Size_Medsize              0.020700
Renew Offer Type_Offer3           0.020342
Policy_Corporate L2               0.019942
Premium_x_Policies                0.019848
Coverage_Premium                  0.018406
Education_High School or Below    0.017685
Policy_Corporate L3               0.014802
Vehicle Size_Small                0.014342
Marital Status_Single             0.011636
dtype: float64


Saved: lasso_feature_importance.png


## 3. RFE Feature Ranking (GradientBoostingRegressor)

In [4]:
n_features_to_select = min(30, len(feature_cols))
gb = GradientBoostingRegressor(n_estimators=100, random_state=RANDOM_STATE)
rfe = RFE(estimator=gb, n_features_to_select=n_features_to_select, step=5)
rfe.fit(X_train_sc, y_train)

rfe_selected = [f for f, s in zip(feature_cols, rfe.support_) if s]
print(f"RFE selected {len(rfe_selected)} features")

# Feature importances from fitted estimator
gb.fit(X_train_sc[:, rfe.support_], y_train)
rfe_importance = pd.Series(gb.feature_importances_, index=rfe_selected).sort_values(ascending=False)
print("\nTop 20 RFE features by importance:")
print(rfe_importance.head(20))

fig, ax = plt.subplots(figsize=(10, 8))
rfe_importance.head(25).sort_values().plot(kind='barh', ax=ax, color='darkorange')
ax.set_title('RFE -- Feature Importances (GradientBoosting)')
ax.set_xlabel('Importance')
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'rfe_feature_importance.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: rfe_feature_importance.png")

RFE selected 30 features



Top 20 RFE features by importance:
Premium_x_Policies               0.457195
Number of Policies               0.380798
Monthly Premium Auto             0.145752
Income                           0.002792
EmploymentStatus_Employed        0.002342
Number of Open Complaints        0.001449
Income_per_Policy                0.001433
Claim_to_Premium_Ratio           0.001322
Months Since Last Claim          0.001160
Policy_Claim_Gap                 0.000946
Total Claim Amount               0.000758
Months Since Policy Inception    0.000712
Coverage_Extended                0.000677
Marital Status_Single            0.000575
Gender_M                         0.000488
Marital Status_Married           0.000406
Response_Yes                     0.000277
EmploymentStatus_Unemployed      0.000135
Policy_Special L3                0.000133
EmploymentStatus_Retired         0.000123
dtype: float64


Saved: rfe_feature_importance.png


## 4. Consensus Feature Set

In [5]:
# Union of LASSO and RFE selections
lasso_set = set(lasso_selected.index.tolist())
rfe_set = set(rfe_selected)
consensus = lasso_set.union(rfe_set)

# Always keep engineered features
engineered = ['Premium_x_Policies', 'Policy_Claim_Gap',
              'Claim_to_Premium_Ratio', 'Income_per_Policy', 'Effective_Month']
for f in engineered:
    if f in feature_cols:
        consensus.add(f)

consensus_list = [f for f in feature_cols if f in consensus]
print(f"Consensus feature set: {len(consensus_list)} features")
print(f"  LASSO only: {len(lasso_set - rfe_set)}")
print(f"  RFE only:   {len(rfe_set - lasso_set)}")
print(f"  Both:       {len(lasso_set & rfe_set)}")

# Save consensus list
import json
consensus_path = PROCESSED_DATA_PATH.parent / 'consensus_features.json'
with open(consensus_path, 'w') as fp:
    json.dump(consensus_list, fp)
print(f"\nSaved consensus feature list -> {consensus_path}")

Consensus feature set: 50 features
  LASSO only: 20
  RFE only:   6
  Both:       24

Saved consensus feature list -> D:\automation\Customer-Lifetime-Value-Prediction-For-AutoInsurance-Company\data\processed\consensus_features.json


## 5. Before vs After Selection -- Model Score Comparison

In [6]:
from sklearn.ensemble import RandomForestRegressor

def quick_score(X_tr, X_te, y_tr, y_te, label):
    rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
    rf.fit(X_tr, y_tr)
    score = r2_score(y_te, rf.predict(X_te))
    print(f"{label}: R2 = {score:.4f}")
    return score

X_tr_all = X_train_sc
X_te_all = X_test_sc

consensus_idx = [list(feature_cols).index(f) for f in consensus_list if f in feature_cols]
X_tr_sel = X_train_sc[:, consensus_idx]
X_te_sel = X_test_sc[:, consensus_idx]

print("Random Forest R2 comparison:")
score_all = quick_score(X_tr_all, X_te_all, y_train, y_test, f"All {len(feature_cols)} features")
score_sel = quick_score(X_tr_sel, X_te_sel, y_train, y_test, f"Consensus {len(consensus_list)} features")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(['All Features', 'Consensus Set'],
       [score_all, score_sel], color=['steelblue', 'seagreen'])
ax.set_ylim(0, 1)
ax.set_ylabel('R2 Score')
ax.set_title('Feature Selection -- Score Comparison')
for i, v in enumerate([score_all, score_sel]):
    ax.text(i, v + 0.01, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
fig.savefig(IMAGES_DIR / 'feature_selection_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print("Saved: feature_selection_comparison.png")

Random Forest R2 comparison:


All 55 features: R2 = 0.9113


Consensus 50 features: R2 = 0.9106
Saved: feature_selection_comparison.png


## Feature Selection Summary

| Method | Features Selected | Criterion |
|---|---|---|
| LASSO (LassoCV) | Non-zero coefficients | Regularization path |
| RFE (GradientBoosting) | Top 30 by recursive elimination | Feature importance rank |
| **Consensus (Union)** | **LASSO ∪ RFE + engineered** | Agreed upon by both methods |

Key insight: Both methods agree that **Monthly Premium Auto**, **Number of Policies**, and **Total Claim Amount** are top predictors of CLV.

---
**Next step: `04_Model_Training.ipynb`**